(genai-02-mm-llm)=
# Model monitoring using LLM

This tutorial illustrates a model monitoring system that leverages LLMs to maintain high standards for deployed models.

**In this tutorial**
- [Prerequisites](#prerequisites)
- [Add the monitoring-function code](#add-the-monitoring-function-code)
- [Deploy the model, enable tracking, and deploy the function](#deploy-the-model-enable-tracking-and-deploy-the-function)

This tutorial explains how an LLM can be monitored. To see it in action, run the [Large Language Model Monitoring](https://github.com/mlrun/demo-monitoring-and-feedback-loop/blob/main/README.md) demo.

## Prerequisites
- GPU node with NVIDIA drivers is necessary for the serving function

In [1]:
import mlrun
from mlrun.features import Feature
from mlrun.datastore.datastore_profile import DatastoreProfileV3io

Create the project

In [2]:
project = mlrun.get_or_create_project("genai-tutorial", user_project=True)
project.set_source(".", pull_at_runtime=True)

> 2025-06-24 14:08:15,605 [info] Project loaded successfully: {"project_name":"genai-tutorial-xingsheng"}


Set the credentials

In [3]:
tsdb_profile = DatastoreProfileV3io(name="v3io-tsdb-profile")
project.register_datastore_profile(tsdb_profile)

stream_profile = DatastoreProfileV3io(
    name="v3io-stream-profile",
    v3io_access_key=mlrun.mlconf.get_v3io_access_key(),
)
project.register_datastore_profile(stream_profile)

In [4]:
project.set_model_monitoring_credentials(
    tsdb_profile_name=tsdb_profile.name,
    stream_profile_name=stream_profile.name,
)

Enable model monitoring for the project

In [5]:
project.enable_model_monitoring(
    base_period=2,  # frequency (in minutes) at which the monitoring applications are triggered
)

> 2025-06-24 14:08:22,163 [warning] enable_model_monitoring: 'base_period' < 10 minutes is not supported in production environments: {"project":"genai-tutorial-xingsheng"}


## Add the monitoring-function code

The monitoring function code collects the traffic to the serving function, analyzes it, and generates results for the specified metric.

In [6]:
%%writefile monit-code.py
import re
from typing import Any, Union

import mlrun
import mlrun.common.schemas
from mlrun.model_monitoring.applications import (
    ModelMonitoringApplicationBase,
    ModelMonitoringApplicationResult,
)

STATUS_RESULT_MAPPING = {
    0: mlrun.common.schemas.model_monitoring.constants.ResultStatusApp.detected,
    1: mlrun.common.schemas.model_monitoring.constants.ResultStatusApp.no_detection,
}


class LLMMonitoringFunction(ModelMonitoringApplicationBase):

    def do_tracking(
        self,
        monitoring_context,
    ) -> Union[
        ModelMonitoringApplicationResult, list[ModelMonitoringApplicationResult]
    ]:
        
        # User monitoring sampling, in this case an integer representing model performance
        # Can be calulated based off the traffic to the function using monitoring_context.sample_df
        result = 0.9

        monitoring_context.log_dataset(
            key="llm-monitoring-df",
            df=monitoring_context.sample_df
        )

        # get status:
        status = STATUS_RESULT_MAPPING[round(result)]

        return ModelMonitoringApplicationResult(
            name="llm_monitoring_df",
            value=result,
            kind=mlrun.common.schemas.model_monitoring.constants.ResultKindApp.model_performance,
            status=status,
            extra_data={},
        )

Overwriting monit-code.py


Define the model monitoring custom function that scans the traffic and calculates the performance metrics

In [7]:
application = project.set_model_monitoring_function(
    func="monit-code.py",
    application_class="LLMMonitoringFunction",
    name="llm-monit",
    image="mlrun/mlrun",
)

In [8]:
application.spec.readiness_timeout = 1200

In [9]:
project.deploy_function(application)

> 2025-06-24 14:08:29,534 [info] Starting remote function deploy
2025-06-24 14:08:29  (info) Deploying function
2025-06-24 14:08:29  (info) Building
2025-06-24 14:08:29  (info) Staging files and preparing base images
2025-06-24 14:08:29  (warn) Using user provided base image, runtime interpreter version is provided by the base image
2025-06-24 14:08:29  (info) Building processor image
2025-06-24 14:10:05  (info) Build complete
2025-06-24 14:10:13  (info) Function deploy complete
> 2025-06-24 14:10:20,618 [info] Model endpoint creation task completed with state succeeded
> 2025-06-24 14:10:20,619 [info] Successfully deployed function: {"external_invocation_urls":[],"internal_invocation_urls":["nuclio-genai-tutorial-xingsheng-llm-monit.default-tenant.svc.cluster.local:8080"]}


DeployStatus(state=ready, outputs={'endpoint': 'http://nuclio-genai-tutorial-xingsheng-llm-monit.default-tenant.svc.cluster.local:8080', 'name': 'genai-tutorial-xingsheng-llm-monit'})

Create a model serving class that loads the LLM and generates responses

In [10]:
%%writefile model-serving.py
import mlrun
from mlrun.serving.v2_serving import V2ModelServer
from transformers import AutoModelForCausalLM, AutoTokenizer
from typing import Any

class LLMModelServer(V2ModelServer):

    def __init__(
        self,
        context: mlrun.MLClientCtx = None,
        name: str = None,
        model_path: str = None,
        model_name: str = None,
        **kwargs
    ):
        super().__init__(name=name, context=context, model_path=model_path, **kwargs)
        self.model_name = model_name
    
    def load(
        self,
    ):
        # Load the model from Hugging Face
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        self.model = AutoModelForCausalLM.from_pretrained(self.model_name)


    def predict(self, request: dict[str, Any]):
        inputs = request.get("inputs", [])
      
        input_ids, attention_mask = self.tokenizer(
            inputs[0], return_tensors="pt"
        ).values()

        outputs = self.model.generate(input_ids=input_ids, attention_mask=attention_mask)

        # Remove input:
        outputs = self.tokenizer.decode(outputs[0])
        outputs = outputs.split(inputs[0])[-1].replace(self.tokenizer.eos_token, "")
        return [{"generated_text": outputs}]

Overwriting model-serving.py


Build an image

In [11]:
commands = [
'pip install pytorch-lightning',
'pip install packaging==21.3',
'pip install transformers adapters openai',
]

In [12]:
# need different version of protobuf to work with different python version, python 3.9 -> protobuf 3.20.2, python 3.11 -> protobuf latest
import sys
minor_version = float(sys.version_info[1])
if minor_version >= 11:    
    commands.append('pip install --upgrade --force-reinstall protobuf')
elif minor_version == 9:
    commands.append('pip install protobuf==3.20.2')
else:
    print(f"minor_version {minor_version} not supported")
commands

['pip install pytorch-lightning',
 'pip install packaging==21.3',
 'pip install transformers adapters openai',
 'pip install --upgrade --force-reinstall protobuf']

Run the following command to build the image, once it's successfully built, no need to run the cell again since it takes quite sometime to build an image

In [13]:
project.build_image(image=".llm-serving-base",
                    base_image='mlrun/mlrun',
                    set_as_default=False, 
                    commands=commands)

> 2025-06-24 14:20:05,104 [info] Started building image: .llm-serving-base


The `overwrite_build_params` parameter default will change from 'False' to 'True' in 1.10.0.


INFO[0000] Retrieving image manifest mlrun/mlrun:1.9.0-rc13 
INFO[0000] Retrieving image mlrun/mlrun:1.9.0-rc13 from registry index.docker.io 
INFO[0000] Built cross stage deps: map[]                
INFO[0000] Retrieving image manifest mlrun/mlrun:1.9.0-rc13 
INFO[0000] Returning cached image manifest              
INFO[0000] Executing 0 build triggers                   
INFO[0000] Building stage 'mlrun/mlrun:1.9.0-rc13' [idx: '0', base-idx: '-1'] 
INFO[0000] Unpacking rootfs as cmd RUN pip install chromadb==0.5.0 langchain==0.2.3 langchain-community==0.2.4 langchain-core==0.2.5 langchain-text-splitters==0.2.1 clean-text==0.6.0 transformers==4.41.2 requires it. 
INFO[0039] ARG STREAM_PROFILE_NAME=$STREAM_PROFILE_NAME 
INFO[0039] ARG TSDB_PROFILE_NAME=$TSDB_PROFILE_NAME     
INFO[0039] ARG datastore-profiles.genai-tutorial-xingsheng.v3io-stream-profile=$datastore-profiles.genai-tutorial-xingsheng.v3io-stream-profile 
INFO[0039] ARG datastore-profiles.genai-tutorial-xingsheng.v3io-tsdb-

BuildStatus(ready=True, outputs={'image': '.llm-serving-base'})

Create the serving function using the class you just defined

In [14]:
serving_fn = project.set_function(
    func="model-serving.py",
    name="llm-server",
    kind="serving",
    image=".llm-serving-base",
)

# Set readiness timeout to 20 minutes, deploy might take a while.
serving_fn.spec.readiness_timeout = 1200

# Attach fuse mount to the function
serving_fn.apply(mlrun.auto_mount())

## Deploy the model, enable tracking, and deploy the function

This tutorial uses the gpt2 model by Google. 

Log the model to the project

In [15]:
base_model = "gpt2"
project.log_model(
    base_model,
    model_file="src/model-iris.pkl",
    inputs=[Feature(value_type="str", name="question")],
    outputs=[Feature(value_type="str", name="answer")],
)

Adding the model parameters to the endpoint. This allow the model server class to initialize.

In [16]:
serving_fn.add_model(
    "gpt2",
    class_name="LLMModelServer",
    model_path=f"store://models/{project.name}/gpt2:latest",
    model_name="gpt2",
)

Enable tracking for the function, then deploy it.

In [17]:
serving_fn.set_tracking()

In [18]:
deployment = serving_fn.deploy()

> 2025-06-24 14:24:27,151 [info] Starting remote function deploy
2025-06-24 14:24:27  (info) Deploying function
2025-06-24 14:24:27  (info) Building
2025-06-24 14:24:27  (info) Staging files and preparing base images
2025-06-24 14:24:27  (warn) Using user provided base image, runtime interpreter version is provided by the base image
2025-06-24 14:24:27  (info) Building processor image
2025-06-24 14:26:52  (info) Build complete
2025-06-24 14:27:34  (info) Function deploy complete
> 2025-06-24 14:27:38,544 [info] Model endpoint creation task completed with state succeeded
> 2025-06-24 14:27:38,545 [info] Successfully deployed function: {"external_invocation_urls":["genai-tutorial-xingsheng-llm-server.default-tenant.app.images-test.iguazio-cd2.com/"],"internal_invocation_urls":["nuclio-genai-tutorial-xingsheng-llm-server.default-tenant.svc.cluster.local:8080"]}


In [19]:
ret = serving_fn.invoke(
    path=f"/v2/models/{base_model}/infer",
    body={"inputs": ["What is a mortgage?"]},
)
ret

{'id': 'c142612e-f618-4461-a9fa-6aa2f4abe8d8',
 'model_name': 'gpt2',
 'outputs': [{'generated_text': '\n\nA mortgage is a loan that is made by a person who is not a resident of the'}],
 'timestamp': '2025-06-24 15:14:56.371875+00:00',
 'model_endpoint_uid': '7bd8688a6c9b4413b9468e98d41cb9d9'}

Test your model serving

Let's generate traffic against the model:

In [20]:
import time


def question_model(questions, serving_function, base_model):
    for question in questions:
        seconds = 0.5
        # Invoking the pretrained model:
        ret = serving_fn.invoke(
            path=f"/v2/models/{base_model}/infer",
            body={"inputs": [question]},
        )
        print(ret)
        time.sleep(seconds)

In [21]:
example_questions = [
    "What is a mortgage?",
    "How does a credit card work?",
    "Who painted the Mona Lisa?",
    "Please plan me a 4-days trip to north Italy",
    "Write me a song",
    "How much people are there in the world?",
    "What is climate change?",
    "How does the stock market work?",
    "Who wrote 'To Kill a Mockingbird'?",
    "Please plan me a 3-day trip to Paris",
    "Write me a poem about the ocean",
    "How many continents are there in the world?",
    "What is artificial intelligence?",
    "How does a hybrid car work?",
    "Who invented the telephone?",
    "Please plan me a week-long trip to New Zealand",
    "What is inflation?",
    "How do vaccines work?",
    "Who discovered gravity?",
    "Please plan me a weekend trip to Tokyo.",
    "Write me a short story about a time traveler.",
    "How many planets are in the solar system?",
    "What is quantum physics?",
    "How does a dishwasher work?",
    "Who wrote '1984'?",
    "Please plan me a 5-day road trip through California.",
    "Write me a haiku about autumn.",
    "What is the tallest mountain in the world?",
    "How does cryptocurrency work?",
    "Who invented the light bulb?",
    "What is the meaning of photosynthesis?",
    "How does an airplane fly?",
    "Who painted 'The Starry Night'?",
    "Please plan me a 10-day trip across South America.",
    "Write me a letter to apologize to a friend.",
    "How many countries are there in the world?",
    "What is renewable energy?",
    "How does Wi-Fi work?",
    "Who directed the movie 'Inception'?",
    "Please plan me a cultural tour of Egypt.",
]

In [22]:
question_model(
    questions=example_questions,
    serving_function=serving_fn,
    base_model=base_model,
)

{'id': '33bb2e66-2163-4c66-8753-31dbdc18f092', 'model_name': 'gpt2', 'outputs': [{'generated_text': '\n\nA mortgage is a loan that is made by a person who is not a resident of the'}], 'timestamp': '2025-06-24 15:15:01.271319+00:00', 'model_endpoint_uid': '7bd8688a6c9b4413b9468e98d41cb9d9'}
{'id': 'f7a952a3-5f76-449b-a548-b64bc21f2b64', 'model_name': 'gpt2', 'outputs': [{'generated_text': '\n\nA credit card is a payment card that is issued by a bank or other financial institution.'}], 'timestamp': '2025-06-24 15:15:02.618338+00:00', 'model_endpoint_uid': '7bd8688a6c9b4413b9468e98d41cb9d9'}
{'id': 'b87de093-4e26-4ef0-9030-6cc4a6a35d0e', 'model_name': 'gpt2', 'outputs': [{'generated_text': '\n\nThe Mona Lisa is a very popular figure in the Marvel Universe. It was first seen'}], 'timestamp': '2025-06-24 15:15:04.129255+00:00', 'model_endpoint_uid': '7bd8688a6c9b4413b9468e98d41cb9d9'}
{'id': 'cd957056-3ca6-4ede-a553-88e564c9c78e', 'model_name': 'gpt2', 'outputs': [{'generated_text': ".\n\nI

Now the traffic to the function is analyzed and the performance is calculated.